# Building an Agent Runtime — Putting Infrastructure Together

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/31_building_an_agent_runtime.ipynb)

## What This Notebook Teaches You

In Notebooks 01–30 we built individual agent capabilities: tool use, memory, planning, reflection, multi-agent coordination. But real-world deployments need **infrastructure** — the plumbing that hosts agents, routes tasks, handles failures, and provides observability.

An **agent runtime** is to agents what an operating system is to processes, or what Kubernetes is to containers. It is the invisible layer that keeps everything running, recoverable, and observable.

**Core analogy**: Think of how a web application server (e.g., Gunicorn, Tomcat) manages HTTP request handlers. It doesn't write the handlers — it instantiates them, routes requests, logs errors, and monitors health. An agent runtime does the same for agents.

By the end of this notebook, you will:

1. **Build an EventBus** — publish-subscribe messaging for inter-agent communication
2. **Build an AgentLogger** — structured logging with full execution traces
3. **Build an AgentRegistry** — register, discover, and manage agent classes
4. **Build an AgentLifecycle manager** — start, stop, restart, and health-check agents
5. **Build a complete AgentRuntime** — the unified infrastructure layer tying it all together
6. **Implement a configuration system** — declarative agent setup with hot-swap support
7. **Run a full demo** — 3 agents coordinating through the runtime with full observability
8. **Compare managed vs. ad-hoc** — quantify the benefits of proper infrastructure

---

> **Prerequisites**: Notebooks 01–12 (agent fundamentals through multi-agent).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~55–70 minutes.

## Part 0: Environment Setup

Same Qwen3-8B setup. The model powers agent behaviors while we build the runtime infrastructure layer around it.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. What Is an Agent Runtime?

### 1.1 — The Infrastructure Layer

Every production system has layers:

```
┌─────────────────────────────────────────────┐
│              Application Logic               │  ← Agent behaviors (ReAct, planning, etc.)
├─────────────────────────────────────────────┤
│              Agent Runtime                   │  ← THIS notebook
│  ┌──────────┬──────────┬──────────┬────────┐│
│  │ EventBus │  Logger  │ Registry │Lifecycle││
│  └──────────┴──────────┴──────────┴────────┘│
├─────────────────────────────────────────────┤
│              Foundation (LLM, Tools)         │  ← Model, tokenizer, tool functions
└─────────────────────────────────────────────┘
```

### 1.2 — Real-World Analogies

| Concept | Operating System | Web Server | **Agent Runtime** |
|---|---|---|---|
| Unit of work | Process | Request handler | Agent |
| Lifecycle | fork/exec/kill | Deploy/undeploy | start/stop/restart |
| Communication | IPC, signals | HTTP, queues | EventBus |
| Observability | syslog, ps | Access logs, metrics | AgentLogger |
| Discovery | PATH, registry | Service registry | AgentRegistry |

### 1.3 — Why Not Just Instantiate Agents Directly?

Without a runtime, you get:
- **No visibility** — which agents ran? What did they do? Where did they fail?
- **No coordination** — agents can't discover or communicate with each other
- **No recovery** — if an agent crashes, nothing restarts it
- **No management** — you can't list, stop, or reconfigure running agents

A runtime solves all of these. Let's build one piece by piece.

## 2. EventBus — Publish-Subscribe Communication

The EventBus lets agents communicate without knowing about each other. An agent publishes an event (e.g., "I found a result"), and any subscriber interested in that event type receives it.

This is the **Observer pattern** applied to agent infrastructure.

In [ ]:
import uuid
from datetime import datetime
from enum import Enum

class EventType(Enum):
    """Standard event types for the agent runtime."""
    AGENT_STARTED = "agent_started"
    AGENT_COMPLETED = "agent_completed"
    AGENT_FAILED = "agent_failed"
    TOOL_CALLED = "tool_called"
    TOOL_RETURNED = "tool_returned"
    TASK_SUBMITTED = "task_submitted"
    TASK_COMPLETED = "task_completed"
    ERROR_OCCURRED = "error_occurred"
    HEALTH_CHECK = "health_check"
    MESSAGE_SENT = "message_sent"

@dataclass
class Event:
    """An event in the runtime system."""
    event_type: EventType
    source: str            # agent or component that emitted the event
    data: Dict[str, Any]   # event payload
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    event_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])

    def __repr__(self):
        return f"Event({self.event_type.value}, src={self.source}, id={self.event_id})"

class EventBus:
    """Publish-subscribe event system for agent communication."""

    def __init__(self):
        self._subscribers: Dict[EventType, List[Callable]] = defaultdict(list)
        self._event_log: List[Event] = []

    def subscribe(self, event_type: EventType, handler: Callable[[Event], None]):
        """Register a handler for a specific event type."""
        self._subscribers[event_type].append(handler)

    def publish(self, event: Event):
        """Publish an event to all subscribers of its type."""
        self._event_log.append(event)
        for handler in self._subscribers.get(event.event_type, []):
            try:
                handler(event)
            except Exception as e:
                error_event = Event(
                    event_type=EventType.ERROR_OCCURRED,
                    source="event_bus",
                    data={"error": str(e), "original_event": event.event_id}
                )
                self._event_log.append(error_event)

    def get_events(self, event_type: Optional[EventType] = None) -> List[Event]:
        """Retrieve logged events, optionally filtered by type."""
        if event_type:
            return [e for e in self._event_log if e.event_type == event_type]
        return list(self._event_log)

    def stats(self) -> Dict[str, int]:
        """Count events by type."""
        counts = defaultdict(int)
        for e in self._event_log:
            counts[e.event_type.value] += 1
        return dict(counts)

# --- Test the EventBus ---
bus = EventBus()
received = []

def on_agent_started(event):
    received.append(event)
    print(f"  Handler received: {event}")

bus.subscribe(EventType.AGENT_STARTED, on_agent_started)
bus.publish(Event(EventType.AGENT_STARTED, source="researcher", data={"task": "search"}))
bus.publish(Event(EventType.AGENT_STARTED, source="writer", data={"task": "draft"}))
bus.publish(Event(EventType.TOOL_CALLED, source="researcher", data={"tool": "search_web"}))

print(f"\nTotal events logged: {len(bus.get_events())}")
print(f"Agent started events: {len(bus.get_events(EventType.AGENT_STARTED))}")
print(f"Event stats: {bus.stats()}")

## 3. AgentLogger — Structured Logging and Execution Traces

Production systems need more than `print()` statements. The AgentLogger provides:
- **Structured logs** with timestamps, levels, and agent IDs
- **Execution traces** — the full history of what each agent did
- **Export capability** — dump traces as JSON for analysis

In [ ]:
class LogLevel(Enum):
    DEBUG = "DEBUG"
    INFO = "INFO"
    WARNING = "WARNING"
    ERROR = "ERROR"
    CRITICAL = "CRITICAL"

LOG_LEVEL_PRIORITY = {
    LogLevel.DEBUG: 0, LogLevel.INFO: 1, LogLevel.WARNING: 2,
    LogLevel.ERROR: 3, LogLevel.CRITICAL: 4
}

@dataclass
class LogEntry:
    """A single log entry."""
    agent_id: str
    level: LogLevel
    message: str
    metadata: Dict[str, Any] = field(default_factory=dict)
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

class AgentLogger:
    """Structured logging system for the agent runtime."""

    def __init__(self, min_level: LogLevel = LogLevel.DEBUG):
        self._traces: Dict[str, List[LogEntry]] = defaultdict(list)
        self._min_level = min_level
        self._all_entries: List[LogEntry] = []

    def log(self, agent_id: str, level: LogLevel, message: str,
            metadata: Optional[Dict[str, Any]] = None):
        """Log a structured entry for an agent."""
        if LOG_LEVEL_PRIORITY[level] < LOG_LEVEL_PRIORITY[self._min_level]:
            return
        entry = LogEntry(
            agent_id=agent_id, level=level,
            message=message, metadata=metadata or {}
        )
        self._traces[agent_id].append(entry)
        self._all_entries.append(entry)

    def get_trace(self, agent_id: str) -> List[LogEntry]:
        """Get the full execution trace for an agent."""
        return self._traces.get(agent_id, [])

    def get_errors(self) -> List[LogEntry]:
        """Get all ERROR and CRITICAL entries across all agents."""
        return [e for e in self._all_entries
                if e.level in (LogLevel.ERROR, LogLevel.CRITICAL)]

    def export_traces(self) -> str:
        """Export all traces as a JSON string."""
        data = {}
        for agent_id, entries in self._traces.items():
            data[agent_id] = [
                {"level": e.level.value, "message": e.message,
                 "metadata": e.metadata, "timestamp": e.timestamp}
                for e in entries
            ]
        return json.dumps(data, indent=2)

    def print_trace(self, agent_id: str):
        """Pretty-print the execution trace for an agent."""
        entries = self.get_trace(agent_id)
        if not entries:
            print(f"No trace for agent '{agent_id}'")
            return
        print(f"\n{'='*60}")
        print(f"  Execution Trace: {agent_id} ({len(entries)} entries)")
        print(f"{'='*60}")
        for e in entries:
            ts = e.timestamp.split("T")[1][:12]
            meta = f" | {e.metadata}" if e.metadata else ""
            print(f"  [{ts}] {e.level.value:<8} {e.message}{meta}")

# --- Test the Logger ---
logger = AgentLogger()
logger.log("researcher", LogLevel.INFO, "Agent started", {"task": "search for AI papers"})
logger.log("researcher", LogLevel.DEBUG, "Calling search tool", {"query": "LLM agents 2024"})
logger.log("researcher", LogLevel.INFO, "Found 5 results")
logger.log("researcher", LogLevel.WARNING, "Result #3 has low relevance score", {"score": 0.23})
logger.log("writer", LogLevel.INFO, "Agent started", {"task": "draft report"})
logger.log("writer", LogLevel.ERROR, "Generation failed — context too long", {"tokens": 8500})

logger.print_trace("researcher")
logger.print_trace("writer")

print(f"\nErrors across all agents: {len(logger.get_errors())}")
print(f"\nJSON export preview:\n{logger.export_traces()[:400]}...")

## 4. AgentRegistry — Agent Discovery and Management

The registry is where agent classes are registered with metadata (name, capabilities, configuration). It acts as a service catalog — other components ask the registry "give me the researcher agent" rather than instantiating classes directly.

In [ ]:
@dataclass
class AgentConfig:
    """Configuration for a registered agent."""
    name: str
    agent_class: type
    capabilities: List[str]
    config: Dict[str, Any] = field(default_factory=dict)
    description: str = ""

class AgentStatus(Enum):
    REGISTERED = "registered"
    RUNNING = "running"
    STOPPED = "stopped"
    FAILED = "failed"
    RESTARTING = "restarting"

@dataclass
class AgentEntry:
    """An entry in the agent registry."""
    config: AgentConfig
    status: AgentStatus = AgentStatus.REGISTERED
    instance: Any = None
    tasks_completed: int = 0
    errors: int = 0

class AgentRegistry:
    """Registry for managing agent classes and instances."""

    def __init__(self):
        self._agents: Dict[str, AgentEntry] = {}

    def register(self, agent_class: type, name: str,
                 capabilities: List[str], config: Optional[Dict] = None,
                 description: str = ""):
        """Register an agent class with metadata."""
        if name in self._agents:
            raise ValueError(f"Agent '{name}' already registered")
        agent_config = AgentConfig(
            name=name, agent_class=agent_class,
            capabilities=capabilities,
            config=config or {}, description=description
        )
        self._agents[name] = AgentEntry(config=agent_config)
        return agent_config

    def get(self, name: str) -> AgentEntry:
        """Get a registered agent entry by name."""
        if name not in self._agents:
            raise KeyError(f"Agent '{name}' not found in registry")
        return self._agents[name]

    def find_by_capability(self, capability: str) -> List[str]:
        """Find agents that have a specific capability."""
        return [
            name for name, entry in self._agents.items()
            if capability in entry.config.capabilities
        ]

    def list_agents(self) -> List[Dict]:
        """List all registered agents with their status."""
        return [
            {"name": name, "status": entry.status.value,
             "capabilities": entry.config.capabilities,
             "tasks_completed": entry.tasks_completed,
             "errors": entry.errors,
             "description": entry.config.description}
            for name, entry in self._agents.items()
        ]

    def update_status(self, name: str, status: AgentStatus):
        """Update the status of a registered agent."""
        self._agents[name].status = status

    def __len__(self):
        return len(self._agents)

# --- Test the Registry ---
# Define some minimal agent classes for demonstration
class BaseAgent:
    def __init__(self, name, config=None):
        self.name = name
        self.config = config or {}
        self.state = "idle"
    def run(self, task):
        return f"[{self.name}] processed: {task}"
    def health(self):
        return self.state != "failed"

class ResearcherAgent(BaseAgent): pass
class AnalystAgent(BaseAgent): pass
class WriterAgent(BaseAgent): pass

registry = AgentRegistry()
registry.register(ResearcherAgent, "researcher",
                  capabilities=["search", "summarize"],
                  description="Searches and summarizes information")
registry.register(AnalystAgent, "analyst",
                  capabilities=["analyze", "compare", "evaluate"],
                  description="Analyzes and compares findings")
registry.register(WriterAgent, "writer",
                  capabilities=["write", "format", "cite"],
                  description="Writes structured reports")

print(f"Registered agents: {len(registry)}")
for agent_info in registry.list_agents():
    print(f"  {agent_info['name']:12s} | {agent_info['status']:10s} | {agent_info['capabilities']}")

print(f"\nAgents with 'analyze' capability: {registry.find_by_capability('analyze')}")
print(f"Agents with 'search' capability:  {registry.find_by_capability('search')}")

## 5. AgentLifecycle — Start, Stop, Restart, Health Check

The lifecycle manager controls the operational state of agents. It instantiates agent classes, monitors their health, and handles graceful shutdown and restart.

```
  REGISTERED ──start()──► RUNNING ──stop()──► STOPPED
       ▲                    │   ▲                │
       │                    │   │                │
       │                 error  restart()        │
       │                    │   │                │
       │                    ▼   │                │
       │                  FAILED                 │
       │                                         │
       └────────── re-register ◄─────────────────┘
```

In [ ]:
class AgentLifecycle:
    """Manages the lifecycle of agents — start, stop, restart, health check."""

    def __init__(self, registry: AgentRegistry, event_bus: EventBus,
                 logger: AgentLogger):
        self.registry = registry
        self.event_bus = event_bus
        self.logger = logger

    def start(self, agent_name: str) -> bool:
        """Instantiate and start an agent."""
        entry = self.registry.get(agent_name)
        if entry.status == AgentStatus.RUNNING:
            self.logger.log(agent_name, LogLevel.WARNING,
                            "Agent already running, skipping start")
            return True
        try:
            # Instantiate the agent class
            instance = entry.config.agent_class(
                name=agent_name, config=entry.config.config
            )
            instance.state = "running"
            entry.instance = instance
            self.registry.update_status(agent_name, AgentStatus.RUNNING)
            self.logger.log(agent_name, LogLevel.INFO,
                            "Agent started successfully",
                            {"class": entry.config.agent_class.__name__})
            self.event_bus.publish(Event(
                EventType.AGENT_STARTED, source=agent_name,
                data={"capabilities": entry.config.capabilities}
            ))
            return True
        except Exception as e:
            self.registry.update_status(agent_name, AgentStatus.FAILED)
            self.logger.log(agent_name, LogLevel.ERROR,
                            f"Failed to start: {e}")
            self.event_bus.publish(Event(
                EventType.ERROR_OCCURRED, source=agent_name,
                data={"error": str(e), "phase": "start"}
            ))
            return False

    def stop(self, agent_name: str) -> bool:
        """Gracefully stop an agent."""
        entry = self.registry.get(agent_name)
        if entry.status != AgentStatus.RUNNING:
            self.logger.log(agent_name, LogLevel.WARNING,
                            f"Cannot stop agent in state: {entry.status.value}")
            return False
        try:
            if entry.instance:
                entry.instance.state = "stopped"
            entry.instance = None
            self.registry.update_status(agent_name, AgentStatus.STOPPED)
            self.logger.log(agent_name, LogLevel.INFO, "Agent stopped")
            return True
        except Exception as e:
            self.logger.log(agent_name, LogLevel.ERROR,
                            f"Error stopping agent: {e}")
            return False

    def restart(self, agent_name: str) -> bool:
        """Stop and restart an agent."""
        self.logger.log(agent_name, LogLevel.INFO, "Restarting agent...")
        self.registry.update_status(agent_name, AgentStatus.RESTARTING)
        entry = self.registry.get(agent_name)
        if entry.instance:
            entry.instance.state = "stopped"
            entry.instance = None
        self.registry.update_status(agent_name, AgentStatus.STOPPED)
        return self.start(agent_name)

    def health_check(self, agent_name: str) -> Dict[str, Any]:
        """Check if an agent is healthy and responsive."""
        entry = self.registry.get(agent_name)
        health = {
            "agent": agent_name,
            "status": entry.status.value,
            "healthy": False,
            "tasks_completed": entry.tasks_completed,
            "errors": entry.errors,
        }
        if entry.status == AgentStatus.RUNNING and entry.instance:
            try:
                health["healthy"] = entry.instance.health()
            except Exception:
                health["healthy"] = False
        self.event_bus.publish(Event(
            EventType.HEALTH_CHECK, source="lifecycle",
            data=health
        ))
        return health

# --- Test the Lifecycle Manager ---
bus2 = EventBus()
logger2 = AgentLogger()
registry2 = AgentRegistry()

registry2.register(ResearcherAgent, "researcher",
                   capabilities=["search"], description="Searches info")
registry2.register(WriterAgent, "writer",
                   capabilities=["write"], description="Writes reports")

lifecycle = AgentLifecycle(registry2, bus2, logger2)

print("Starting agents...")
lifecycle.start("researcher")
lifecycle.start("writer")

print("\nHealth checks:")
for name in ["researcher", "writer"]:
    h = lifecycle.health_check(name)
    print(f"  {name}: healthy={h['healthy']}, status={h['status']}")

print("\nRestarting researcher...")
lifecycle.restart("researcher")

print("\nFinal status:")
for agent_info in registry2.list_agents():
    print(f"  {agent_info['name']:12s} | {agent_info['status']}")

## 6. The Agent Runtime — Tying Everything Together

Now we combine all four components into a single `AgentRuntime` class. This is the top-level interface that users interact with.

```
              ┌──────────────────────────────────────────┐
              │            AgentRuntime                   │
              │                                          │
              │  submit_task(agent, task)                 │
              │  get_status() → dashboard                │
              │  register_agent(...)                      │
              │  start_agent(...) / stop_agent(...)       │
              │                                          │
              │  ┌──────────┐  ┌──────────┐             │
              │  │ EventBus │  │  Logger  │             │
              │  └──────────┘  └──────────┘             │
              │  ┌──────────┐  ┌──────────┐             │
              │  │ Registry │  │Lifecycle │             │
              │  └──────────┘  └──────────┘             │
              └──────────────────────────────────────────┘
```

In [ ]:
class AgentRuntime:
    """The complete agent runtime — infrastructure for hosting and managing agents."""

    def __init__(self, name: str = "default"):
        self.name = name
        self.event_bus = EventBus()
        self.logger = AgentLogger()
        self.registry = AgentRegistry()
        self.lifecycle = AgentLifecycle(
            self.registry, self.event_bus, self.logger
        )
        self._task_count = 0
        self._start_time = datetime.now().isoformat()

        # Subscribe to key events for internal tracking
        self.event_bus.subscribe(EventType.ERROR_OCCURRED, self._on_error)

    def _on_error(self, event: Event):
        """Internal handler for error events."""
        self.logger.log("runtime", LogLevel.ERROR,
                        f"Error from {event.source}: {event.data.get('error', 'unknown')}")

    def register_agent(self, agent_class: type, name: str,
                       capabilities: List[str], config: Optional[Dict] = None,
                       description: str = "", auto_start: bool = True):
        """Register and optionally start an agent."""
        self.registry.register(
            agent_class, name, capabilities,
            config=config, description=description
        )
        self.logger.log("runtime", LogLevel.INFO,
                        f"Registered agent: {name}",
                        {"capabilities": capabilities})
        if auto_start:
            self.lifecycle.start(name)

    def submit_task(self, agent_name: str, task: str) -> Dict[str, Any]:
        """Submit a task to a specific agent and return the result."""
        self._task_count += 1
        task_id = f"task-{self._task_count:04d}"

        self.event_bus.publish(Event(
            EventType.TASK_SUBMITTED, source="runtime",
            data={"task_id": task_id, "agent": agent_name, "task": task}
        ))
        self.logger.log(agent_name, LogLevel.INFO,
                        f"Task submitted: {task_id}",
                        {"task_preview": task[:80]})

        entry = self.registry.get(agent_name)
        if entry.status != AgentStatus.RUNNING or not entry.instance:
            error_msg = f"Agent '{agent_name}' is not running (status: {entry.status.value})"
            self.logger.log(agent_name, LogLevel.ERROR, error_msg)
            return {"task_id": task_id, "status": "error", "error": error_msg}

        try:
            start_t = time.time()
            result = entry.instance.run(task)
            elapsed = time.time() - start_t

            entry.tasks_completed += 1
            self.logger.log(agent_name, LogLevel.INFO,
                            f"Task completed: {task_id}",
                            {"elapsed_s": round(elapsed, 3)})
            self.event_bus.publish(Event(
                EventType.TASK_COMPLETED, source=agent_name,
                data={"task_id": task_id, "elapsed_s": round(elapsed, 3)}
            ))
            return {"task_id": task_id, "status": "completed",
                    "result": result, "elapsed_s": round(elapsed, 3)}

        except Exception as e:
            entry.errors += 1
            self.logger.log(agent_name, LogLevel.ERROR,
                            f"Task failed: {task_id} — {e}")
            self.event_bus.publish(Event(
                EventType.ERROR_OCCURRED, source=agent_name,
                data={"task_id": task_id, "error": str(e)}
            ))
            return {"task_id": task_id, "status": "error", "error": str(e)}

    def get_status(self) -> Dict[str, Any]:
        """Get a complete overview of the runtime state."""
        agents = self.registry.list_agents()
        return {
            "runtime_name": self.name,
            "started_at": self._start_time,
            "total_agents": len(self.registry),
            "running_agents": sum(1 for a in agents if a["status"] == "running"),
            "total_tasks": self._task_count,
            "total_events": len(self.event_bus.get_events()),
            "total_errors": len(self.logger.get_errors()),
            "agents": agents,
            "event_stats": self.event_bus.stats(),
        }

    def stop_all(self):
        """Stop all running agents."""
        for agent_info in self.registry.list_agents():
            if agent_info["status"] == "running":
                self.lifecycle.stop(agent_info["name"])
        self.logger.log("runtime", LogLevel.INFO, "All agents stopped")

print("✓ AgentRuntime class defined")

### 6.1 — Quick Smoke Test

Let's make sure all the pieces fit together before building the full demo.

In [ ]:
# Quick integration test
rt = AgentRuntime(name="test-runtime")
rt.register_agent(ResearcherAgent, "researcher",
                  capabilities=["search", "summarize"],
                  description="Finds and summarizes information")
rt.register_agent(WriterAgent, "writer",
                  capabilities=["write", "format"],
                  description="Produces written output")

# Submit tasks
r1 = rt.submit_task("researcher", "Find papers on agent architectures")
r2 = rt.submit_task("writer", "Write a summary of agent architectures")

print(f"Task 1: {r1['status']} — {r1['result']}")
print(f"Task 2: {r2['status']} — {r2['result']}")

# Check status
status = rt.get_status()
print(f"\nRuntime: {status['runtime_name']}")
print(f"  Agents: {status['running_agents']}/{status['total_agents']} running")
print(f"  Tasks processed: {status['total_tasks']}")
print(f"  Events fired: {status['total_events']}")
print(f"  Errors: {status['total_errors']}")

rt.stop_all()

## 7. Configuration System — Declarative Agent Setup

Production runtimes don't hard-code agent setup. They use **configuration files** (or dictionaries) that declare which agents to run and how to configure them.

This enables:
- **Hot-swapping** — change agent config without changing code
- **Environment-specific** — different configs for dev, staging, production
- **Reproducibility** — the config fully describes the runtime state

In [ ]:
@dataclass
class RuntimeConfig:
    """Declarative configuration for an agent runtime."""
    runtime_name: str
    agents: List[Dict[str, Any]]
    global_settings: Dict[str, Any] = field(default_factory=dict)

def build_runtime_from_config(config: RuntimeConfig,
                              agent_classes: Dict[str, type]) -> AgentRuntime:
    """Build a fully configured runtime from a declarative config."""
    runtime = AgentRuntime(name=config.runtime_name)

    for agent_def in config.agents:
        cls_name = agent_def["class"]
        if cls_name not in agent_classes:
            runtime.logger.log("config", LogLevel.ERROR,
                               f"Unknown agent class: {cls_name}")
            continue
        runtime.register_agent(
            agent_class=agent_classes[cls_name],
            name=agent_def["name"],
            capabilities=agent_def.get("capabilities", []),
            config=agent_def.get("config", {}),
            description=agent_def.get("description", ""),
            auto_start=agent_def.get("auto_start", True)
        )

    runtime.logger.log("config", LogLevel.INFO,
                        f"Runtime built from config: {len(config.agents)} agents",
                        {"global": config.global_settings})
    return runtime

# Define a configuration
config = RuntimeConfig(
    runtime_name="castalia-scholar",
    agents=[
        {
            "name": "researcher", "class": "ResearcherAgent",
            "capabilities": ["search", "summarize", "extract"],
            "description": "Searches for and extracts information from sources",
            "config": {"max_sources": 10, "search_depth": 3},
        },
        {
            "name": "analyst", "class": "AnalystAgent",
            "capabilities": ["analyze", "compare", "evaluate", "synthesize"],
            "description": "Analyzes research findings and identifies patterns",
            "config": {"min_confidence": 0.7},
        },
        {
            "name": "writer", "class": "WriterAgent",
            "capabilities": ["write", "format", "cite", "revise"],
            "description": "Produces structured research reports",
            "config": {"style": "academic", "max_sections": 8},
        },
    ],
    global_settings={"log_level": "INFO", "max_concurrent_tasks": 5}
)

# Build runtime from config
CLASS_MAP = {
    "ResearcherAgent": ResearcherAgent,
    "AnalystAgent": AnalystAgent,
    "WriterAgent": WriterAgent,
}
runtime = build_runtime_from_config(config, CLASS_MAP)

status = runtime.get_status()
print(f"Runtime '{status['runtime_name']}' built from config:")
for a in status["agents"]:
    print(f"  {a['name']:12s} | {a['status']:10s} | {a['capabilities']}")

## 8. Full Demo — LLM-Powered Agents in the Runtime

Now let's build agents that actually use the LLM. Each agent has a specialized system prompt and uses the `generate()` function. The runtime manages their lifecycle and logs everything.

In [ ]:
class LLMAgent(BaseAgent):
    """Base agent that uses the LLM for processing."""

    def __init__(self, name, config=None):
        super().__init__(name, config)
        self.system_prompt = config.get("system_prompt", "You are a helpful assistant.")
        self.max_tokens = config.get("max_tokens", 300)

    def run(self, task):
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": task}
        ]
        return generate(messages, max_new_tokens=self.max_tokens, temperature=0.7)

class LLMResearcher(LLMAgent):
    """Research agent that extracts key information from a topic."""
    pass

class LLMAnalyst(LLMAgent):
    """Analysis agent that evaluates and compares findings."""
    pass

class LLMWriter(LLMAgent):
    """Writer agent that produces structured text."""
    pass

# Build a runtime with LLM agents
llm_config = RuntimeConfig(
    runtime_name="castalia-llm-demo",
    agents=[
        {
            "name": "researcher", "class": "LLMResearcher",
            "capabilities": ["search", "summarize"],
            "description": "Finds and summarizes key information",
            "config": {
                "system_prompt": "You are a research assistant. Given a topic, identify the 3 most important facts or findings. Be concise — use bullet points.",
                "max_tokens": 250,
            },
        },
        {
            "name": "analyst", "class": "LLMAnalyst",
            "capabilities": ["analyze", "compare"],
            "description": "Analyzes and evaluates findings",
            "config": {
                "system_prompt": "You are an analytical assistant. Given research findings, evaluate their significance, identify patterns, and note any contradictions. Be structured and precise.",
                "max_tokens": 300,
            },
        },
        {
            "name": "writer", "class": "LLMWriter",
            "capabilities": ["write", "format"],
            "description": "Writes cohesive summaries",
            "config": {
                "system_prompt": "You are a technical writer. Given analysis results, write a clear, well-structured summary paragraph. Use professional tone.",
                "max_tokens": 300,
            },
        },
    ]
)

LLM_CLASS_MAP = {
    "LLMResearcher": LLMResearcher,
    "LLMAnalyst": LLMAnalyst,
    "LLMWriter": LLMWriter,
}
llm_runtime = build_runtime_from_config(llm_config, LLM_CLASS_MAP)

print(f"✓ LLM Runtime ready: {len(llm_runtime.registry)} agents")

### 8.1 — Running the Pipeline

We submit a research topic, pass results through the pipeline (Researcher → Analyst → Writer), and observe the full execution through the runtime's logging.

In [ ]:
topic = "The impact of chain-of-thought prompting on LLM reasoning"

print("=" * 60)
print(f"  PIPELINE: {topic}")
print("=" * 60)

# Step 1: Research
print("\n▸ Step 1: Researcher")
research_result = llm_runtime.submit_task("researcher", f"Research this topic and list key findings: {topic}")
research_output = research_result["result"]
print(f"  Status: {research_result['status']} ({research_result['elapsed_s']:.1f}s)")
print(f"  Output preview: {research_output[:200]}...")

# Step 2: Analyze
print("\n▸ Step 2: Analyst")
analysis_result = llm_runtime.submit_task("analyst", f"Analyze these research findings:\n{research_output}")
analysis_output = analysis_result["result"]
print(f"  Status: {analysis_result['status']} ({analysis_result['elapsed_s']:.1f}s)")
print(f"  Output preview: {analysis_output[:200]}...")

# Step 3: Write
print("\n▸ Step 3: Writer")
write_result = llm_runtime.submit_task("writer", f"Write a summary based on this analysis:\n{analysis_output}")
write_output = write_result["result"]
print(f"  Status: {write_result['status']} ({write_result['elapsed_s']:.1f}s)")
print(f"\n{'─'*60}")
print("  FINAL OUTPUT:")
print(f"{'─'*60}")
print(write_output)

## 9. Observability Dashboard

A key benefit of the runtime is **full visibility** into what happened. Let's build a dashboard that summarizes the entire execution.

In [ ]:
def print_dashboard(runtime: AgentRuntime):
    """Print a formatted observability dashboard."""
    status = runtime.get_status()

    print("╔" + "═"*58 + "╗")
    print(f"║  RUNTIME DASHBOARD: {status['runtime_name']:36s} ║")
    print("╠" + "═"*58 + "╣")

    # Agents overview
    print("║  AGENTS" + " "*50 + "║")
    print("║  " + "─"*54 + "  ║")
    for a in status["agents"]:
        status_icon = "🟢" if a["status"] == "running" else "🔴" if a["status"] == "failed" else "⚪"
        line = f"  {status_icon} {a['name']:12s} | {a['status']:10s} | tasks: {a['tasks_completed']} | errs: {a['errors']}"
        print(f"║{line:58s}║")

    # Task summary
    print("╠" + "═"*58 + "╣")
    print(f"║  TASKS:  {status['total_tasks']:5d} submitted" + " "*30 + "║")
    print(f"║  EVENTS: {status['total_events']:5d} fired" + " "*33 + "║")
    print(f"║  ERRORS: {status['total_errors']:5d} logged" + " "*32 + "║")

    # Event breakdown
    print("╠" + "═"*58 + "╣")
    print("║  EVENT BREAKDOWN" + " "*41 + "║")
    print("║  " + "─"*54 + "  ║")
    for evt_type, count in status["event_stats"].items():
        line = f"  {evt_type:30s} : {count:4d}"
        print(f"║{line:58s}║")

    print("╚" + "═"*58 + "╝")

print_dashboard(llm_runtime)

### 9.1 — Execution Traces

The logger captured every step. Let's view the trace for each agent — this is what makes debugging multi-agent systems feasible.

In [ ]:
for agent_name in ["researcher", "analyst", "writer"]:
    llm_runtime.logger.print_trace(agent_name)

# Also show the runtime-level log
llm_runtime.logger.print_trace("runtime")

# Export full trace as JSON
trace_json = llm_runtime.logger.export_traces()
print(f"\n\nFull trace JSON size: {len(trace_json)} chars")
print(f"Preview: {trace_json[:300]}...")

## 10. Error Handling and Recovery

What happens when things go wrong? The runtime should catch errors, log them, and allow recovery. Let's simulate a failure scenario.

In [ ]:
class FailingAgent(BaseAgent):
    """An agent that fails on certain inputs — for testing error handling."""
    def run(self, task):
        if "fail" in task.lower():
            raise RuntimeError("Simulated agent failure!")
        return f"[{self.name}] processed: {task}"

# Register the failing agent
llm_runtime.registry.register(FailingAgent, "faulty",
                               capabilities=["test"],
                               description="Agent that sometimes fails")
llm_runtime.lifecycle.start("faulty")

# Submit a task that will succeed
r_ok = llm_runtime.submit_task("faulty", "Normal task, no problems")
print(f"Normal task: {r_ok['status']} — {r_ok.get('result', r_ok.get('error'))}")

# Submit a task that will fail
r_fail = llm_runtime.submit_task("faulty", "Please fail on this task")
print(f"Failing task: {r_fail['status']} — {r_fail.get('error')}")

# Submit another normal task — agent should still work
r_after = llm_runtime.submit_task("faulty", "Task after failure")
print(f"After failure: {r_after['status']} — {r_after.get('result', r_after.get('error'))}")

# Check health
health = llm_runtime.lifecycle.health_check("faulty")
print(f"\nHealth: {health}")
print(f"Faulty agent stats — tasks: {llm_runtime.registry.get('faulty').tasks_completed}, errors: {llm_runtime.registry.get('faulty').errors}")

## 11. Comparison: Ad-Hoc vs. Managed Runtime

| Dimension | Ad-Hoc Agent Creation | Managed Agent Runtime |
|---|---|---|
| **Instantiation** | Manual: `agent = MyAgent()` | Declarative: config → registry → lifecycle |
| **Discovery** | Hard-coded references | Registry with capability-based lookup |
| **Communication** | Direct method calls | EventBus with pub/sub decoupling |
| **Logging** | Scattered `print()` | Structured AgentLogger with traces |
| **Error handling** | try/except per call | Centralized with automatic logging |
| **Health monitoring** | None | health_check() with status tracking |
| **Recovery** | Manual restart | lifecycle.restart() with state management |
| **Configuration** | Code changes | Hot-swap via RuntimeConfig |
| **Observability** | None | Full dashboard with event stats |
| **Debugging** | Read source code | Trace replay, event logs, error history |
| **Scaling** | Rewrite everything | Add agents via config |

**Bottom line**: A runtime adds ~200 lines of infrastructure code but provides **10x better visibility and reliability** for any multi-agent system beyond a prototype.

### 11.1 — Event Replay for Debugging

One of the most powerful debugging techniques is **event replay** — you can reconstruct exactly what happened by replaying the event log.

In [ ]:
def replay_events(runtime: AgentRuntime, event_type: Optional[EventType] = None):
    """Replay events chronologically for debugging."""
    events = runtime.event_bus.get_events(event_type)
    print(f"\nEvent Replay ({len(events)} events):")
    print("─" * 70)
    for i, event in enumerate(events):
        ts = event.timestamp.split("T")[1][:12]
        data_preview = str(event.data)[:60]
        print(f"  {i+1:3d}. [{ts}] {event.event_type.value:20s} | {event.source:12s} | {data_preview}")
    print("─" * 70)

# Replay all events
replay_events(llm_runtime)

# Replay only task events
print("\n\n--- Task events only ---")
replay_events(llm_runtime, EventType.TASK_SUBMITTED)
replay_events(llm_runtime, EventType.TASK_COMPLETED)

## 12. Summary and Key Takeaways

### What We Built

| Component | Purpose | Key Methods |
|---|---|---|
| **EventBus** | Pub/sub communication | publish(), subscribe(), get_events() |
| **AgentLogger** | Structured logging + traces | log(), get_trace(), export_traces() |
| **AgentRegistry** | Agent discovery + management | register(), get(), find_by_capability() |
| **AgentLifecycle** | Operational control | start(), stop(), restart(), health_check() |
| **AgentRuntime** | Unified infrastructure | submit_task(), get_status(), register_agent() |
| **RuntimeConfig** | Declarative setup | build_runtime_from_config() |

### Design Principles

1. **Separation of concerns** — each component does one thing well
2. **Loose coupling** — components communicate through events, not direct calls
3. **Observability by default** — every action is logged and traceable
4. **Declarative configuration** — describe what you want, not how to build it
5. **Graceful degradation** — errors are caught, logged, and recoverable

### What's Next

- **Notebook 32**: Agent-to-Agent Communication Protocols — structured message passing
- **Notebook 33**: Evaluation and Testing — how to systematically test agent systems
- **Notebook 34**: Castalia Scholar capstone — Research Agent
- **Notebook 35**: Castalia Scholar capstone — Writing Agent

---

*You now have the infrastructure to build production-quality agent systems. The agents from earlier notebooks can be plugged into this runtime for full lifecycle management and observability.*